# Halo interpolation

Where no patch of the same level covers a halo point, `exchange_halos` interpolates it from
the level below. Bilinear interpolation, the default, has an error of order $h^2$ in the
value. A stencil that only takes first differences across the halo (a gradient) stays
consistent with it, but a second difference divides by $h^2$: the 5-point Laplacian at the
nodes on a coarse-fine edge has an error of order one, which does not go away as the mesh is
refined. In MHD with the potential $A_z$ on the nodes, that Laplacian is the current
$J_z = -\nabla^2 A_z$.

`exchange_halos(name, interpolation="quadratic")` takes three coarse points per axis instead
of two (biquadratic in 2D, triquadratic in 3D, one-sided next to a wall). The halo values are
then third order, and the Laplacian at the edge first order. The patch's own points, among
them the nodes it shares with the level below, are not changed by either option.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from patchwork import AMRHierarchy

## The Laplacian at a coarse-fine edge

A smooth potential on the nodes of a base level and of one patch in its middle, set on the
patches' own nodes only. After the halo exchange, the 5-point Laplacian is taken on every
node of the patch and compared with the exact one.

In [ ]:
def potential(x, y):
    return np.sin(2.0 * np.pi * x + 0.3) * np.cos(2.0 * np.pi * y - 0.2)


def laplacian_error(n: int, interpolation: str) -> np.ndarray:
    # the error of the 5-point Laplacian on the nodes of the fine patch
    mesh = AMRHierarchy(base_shape=(n, n))
    fine = mesh.add_patch(level=1, box=((n // 2, 3 * n // 2),) * 2)
    mesh.allocate_field("A", halo=1, centering="node")
    for patch in mesh:
        x, y = patch.local_node_coordinates(mesh, "A")
        patch.field("A").local[...] = potential(x, y)
    mesh.exchange_halos("A", interpolation=interpolation)
    u = np.asarray(fine.field("A").local_with_halos)
    h = mesh.spacing(1)[0]
    laplacian = (
        u[2:, 1:-1] + u[:-2, 1:-1] + u[1:-1, 2:] + u[1:-1, :-2] - 4.0 * u[1:-1, 1:-1]
    ) / h**2
    x, y = fine.local_node_coordinates(mesh, "A")
    return np.abs(laplacian + 8.0 * np.pi**2 * potential(x, y))


n = 32
fig, axes = plt.subplots(2, 1, figsize=(5.5, 9.0))
for ax, interpolation in zip(axes, ("bilinear", "quadratic"), strict=True):
    error = laplacian_error(n, interpolation)
    image = ax.imshow(error.T, origin="lower", extent=(0.25, 0.75, 0.25, 0.75))
    fig.colorbar(image, ax=ax, label="error of the Laplacian")
    ax.set_title(f"{interpolation}: largest error {error.max():.2f}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
fig.tight_layout()

With bilinear halos the error sits on the edge nodes and is of the size of the Laplacian
itself ($8\pi^2 \approx 79$ times the potential), largest at the halo nodes half way between
two coarse nodes; inside the patch, where the stencil sees only own nodes, it is the small
truncation error of the 5-point stencil. With quadratic halos the edge error is twenty
times smaller here, and it halves with every refinement.

## Convergence

The largest error over the edge nodes, as the base level is refined:

In [ ]:
sizes = np.array([8, 16, 32, 64, 128])
errors = {
    interpolation: np.array([laplacian_error(n, interpolation).max() for n in sizes])
    for interpolation in ("bilinear", "quadratic")
}
print("   n    bilinear   quadratic")
for row in zip(sizes, errors["bilinear"], errors["quadratic"], strict=True):
    print(f"{row[0]:4d}   {row[1]:9.3f}   {row[2]:9.3f}")
order = np.log2(errors["quadratic"][-2] / errors["quadratic"][-1])
print(f"order of the quadratic halos: {order:.2f}")

h = 1.0 / (2 * sizes)
fig, ax = plt.subplots(figsize=(5.5, 4.0))
for interpolation, marker in (("bilinear", "o"), ("quadratic", "s")):
    ax.loglog(h, errors[interpolation], marker=marker, label=interpolation)
ax.loglog(h, errors["quadratic"][0] * h / h[0], "k--", label="first order")
ax.set_xticks(h, [f"1/{2 * n}" for n in sizes])
ax.minorticks_off()
ax.set_xlabel("fine spacing $h$")
ax.set_ylabel("largest error of the Laplacian")
ax.legend()
fig.tight_layout()

## When to use it

The quadratic halos cost three coarse points per axis instead of two (9 instead of 4 in 2D,
27 instead of 8 in 3D) and have their own schedule, built at the first call. Use them for a
field whose second differences matter across a coarse-fine edge: a node potential whose
Laplacian is a current or a charge density. For fields that only enter through first
differences, or that must stay within the range of the coarse values (a density: the
quadratic interpolant can overshoot), keep the default. Where a coarse point of the 3 x 3
stencil is on no patch, the halo point falls back to the bilinear weights.